In [ ]:
# ---------------------------------------------------------------------------
# CELL 0 - First look at the dataset's participant table.
#
# Goal: find out what is actually available before committing to any design.
# ds000030 documents 272 participants across four diagnostic groups, but the
# modality columns record which SCANS EXIST, not which scans are usable.
# ---------------------------------------------------------------------------

import pandas as pd
df = pd.read_csv('participants.tsv', sep='\t')

# 1. Diagnosis counts
print(df['diagnosis'].value_counts(), '\n')

# 2. Prefix rule check - id contains diagnosis information
print(pd.crosstab(df['participant_id'].str[4:6], df['diagnosis']), '\n')

# 3. What the modality flags actually contain
#    Critical: is `dwi` a boolean, a count, a filename, or NaN-vs-present?
print("dwi column raw values:", df['dwi'].unique())
print("T1w column raw values:", df['T1w'].unique(), '\n')

# 4. The number that matters: who has both DWI and T1
# Note: we abandon T1 later, see 2.6 of Report
has_dwi = df['dwi'].notna()
has_t1  = df['T1w'].notna()
print(pd.crosstab(df['diagnosis'], has_dwi & has_t1), '\n')

# 5. Artifact flag distribution
#    The curators flagged T1 aliasing (~20% of scans) in `ghost_NoGhost`.
#    KEY: the artifact is not evenly spread
#    across groups (40% SCHZ vs 15% control). A confound correlated with
#    diagnosis cannot be adjusted away after the fact, so it had to be
#    designed around rather than corrected. See REPORT.md 2.6.
print(pd.crosstab(df['diagnosis'], df['ghost_NoGhost']))

In [ ]:
# ---------------------------------------------------------------------------
# CELL 1 - Feasibility: how big a cohort is possible, and what does each
# eligibility requirement cost?
#
# BEFORE downloading. Each participant is ~100 MB of diffusion data
# ---------------------------------------------------------------------------

import pandas as pd
df = pd.read_csv('participants.tsv', sep='\t')

sub = df[df['diagnosis'].isin(['CONTROL','SCHZ'])].copy()

# Which requirement costs what?
print(pd.crosstab(sub['diagnosis'], sub['dwi'].notna(),  rownames=['dx'], colnames=['has_dwi']), '\n')
print(pd.crosstab(sub['diagnosis'], sub['T1w'].notna(), rownames=['dx'], colnames=['has_T1']), '\n')

# Cohort on DWI alone
coh = sub[sub['dwi'].notna()].copy()
print("Eligible on DWI only:")
print(coh['diagnosis'].value_counts(), '\n')

# Can we even match? Age and sex distributions
# age
print(coh.groupby('diagnosis')['age'].describe()[['count','mean','std','min','max']], '\n')
# gender
print(pd.crosstab(coh['diagnosis'], coh['gender']))

In [3]:
# ---------------------------------------------------------------------------
# CELL 2 - FIRST cohort attempt: match on sex and age only.
#
# THIS COHORT WAS LATER DISCARDED. It is retained because the reason it failed
# is the central methodological finding of the project: it looks perfectly
# balanced on every variable examined here, and is nonetheless unusable
# (see cells 3-4).
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np
from scipy import stats

df = pd.read_csv('participants.tsv', sep='\t')
coh = df[df['diagnosis'].isin(['CONTROL','SCHZ']) & df['dwi'].notna()].copy()

scz  = coh[coh['diagnosis']=='SCHZ'].copy()
ctrl = coh[coh['diagnosis']=='CONTROL'].copy()

# Match hardest cases first: extreme ages have fewest candidates
scz = scz.reindex(scz['age'].sub(scz['age'].mean()).abs().sort_values(ascending=False).index)

used, pairs = set(), []
for _, p in scz.iterrows():
    # Exact match on sex; `used` enforces one control per patient (matching
    # without replacement, so no control is counted twice).
    cand = ctrl[(ctrl['gender']==p['gender']) & (~ctrl['participant_id'].isin(used))]
    if cand.empty:
        print("NO MATCH for", p['participant_id'], p['gender'], p['age']); continue
    # Nearest available age among valid candidates.
    m = cand.iloc[(cand['age']-p['age']).abs().argmin()]
    used.add(m['participant_id'])
    pairs.append({'scz_id':p['participant_id'],'scz_age':p['age'],
                  'ctrl_id':m['participant_id'],'ctrl_age':m['age'],
                  'gender':p['gender'],'diff':abs(p['age']-m['age'])})

pairs = pd.DataFrame(pairs)
print(f"\nmatched {len(pairs)} pairs")
# Match quality: a large maximum age gap would mean the pool ran thin
# somewhere, even if the group means end up comparable.
print("age diff within pair: mean %.2f  max %d" % (pairs['diff'].mean(), pairs['diff'].max()))
print((pairs['diff']>5).sum(), "pairs differ by >5 years\n")

sel = coh[coh['participant_id'].isin(pairs['scz_id']) | coh['participant_id'].isin(pairs['ctrl_id'])].copy()
print(sel.groupby('diagnosis')['age'].agg(['count','mean','std']), '\n')
print(pd.crosstab(sel['diagnosis'], sel['gender']), '\n')

# Balance checks on what was matched. These all pass
a = sel[sel['diagnosis']=='SCHZ']['age']; b = sel[sel['diagnosis']=='CONTROL']['age']
print("age t-test  p = %.3f" % stats.ttest_ind(a,b).pvalue)
print("sex chi2    p = %.3f" % stats.chi2_contingency(pd.crosstab(sel['diagnosis'], sel['gender']))[1])
print(pd.crosstab(sel['diagnosis'], sel['ghost_NoGhost']))

sel.to_csv('cohort.tsv', sep='\t', index=False)
pairs.to_csv('matched_pairs.tsv', sep='\t', index=False)


matched 49 pairs
age diff within pair: mean 0.65  max 3
0 pairs differ by >5 years

           count       mean       std
diagnosis                            
CONTROL       49  35.755102  8.511097
SCHZ          49  36.244898  8.837728 

gender      F   M
diagnosis        
CONTROL    12  37
SCHZ       12  37 

age t-test  p = 0.781
sex chi2    p = 1.000
ghost_NoGhost  No_ghost  ghost
diagnosis                     
CONTROL              43      6
SCHZ                 29     20


In [4]:
# ---------------------------------------------------------------------------
# CELL 3 - Extract acquisition parameters from the BIDS JSON sidecars.
#
# Every BIDS scan carries a .json sidecar describing how it was acquired. The
# dataset documentation says nothing about acquisition heterogeneity, so this
# is the only way to find out whether all scans were collected the same way.
#
# we need to match 2 disjoint sets of columns
#
# Writes acq_params.tsv, which is the evidence base for TWO separate claims:
#   - `serial`, `tr`, `nslices` -> the protocol confound (cell 4, REPORT 2.2)
#   - `pe`, `trt`               -> phase encoding is uniform, so no
#                                  susceptibility correction is possible
#                                  (REPORT 2.5). Nothing else records these.
# sub  phase encoding direction; te - echo time
# serial	DeviceSerialNumber	Which physical scanner
# tr	RepetitionTime	Time between excitation pulses
# nslices	length of SliceTiming	Slices per volume
# pe	PhaseEncodingDirection	Axis along which EPI distortion falls
# trt	TotalReadoutTime	How long the EPI readout takes

# ---------------------------------------------------------------------------

import json, pandas as pd
rows = []
for s in pd.read_csv('cohort.tsv', sep='\t')['participant_id']:
    d = json.load(open(f'raw/{s}/dwi/{s}_dwi.json'))
    rows.append({'sub': s, 'pe': d.get('PhaseEncodingDirection'),
                 'trt': round(d.get('TotalReadoutTime', float('nan')), 6),
                 'te': d.get('EchoTime'), 'tr': d.get('RepetitionTime'),
                 'nslices': len(d.get('SliceTiming', [])),
                 'serial': d.get('DeviceSerialNumber')})
p = pd.DataFrame(rows)
for c in ['pe','trt','te','tr','nslices','serial']:
    print(c, p[c].value_counts().to_dict())
p.to_csv('acq_params.tsv', sep='\t', index=False)

pe {'j-': 98}
trt {0.088318: 98}
te {0.0926: 66, 0.093: 32}
tr {8.4: 53, 9.0: 32, 7.0: 13}
nslices {60: 85, 50: 13}
serial {'35343': 66, '35426': 32}


In [6]:
# ---------------------------------------------------------------------------
# CELL 4 - Test acquisition protocol against diagnosis.
#
# Scanner hardware differs in noise, gradient calibration and reconstruction,
# all of which shift measured FA. If scanner assignment correlates with
# diagnosis, then "patients have lower FA" and "scanner 35426 gives lower FA"
# predict IDENTICAL data. The two variables carry the same information, so no
# post-hoc covariate can separate them. The only remedy is cohort design.
# ---------------------------------------------------------------------------

import pandas as pd
p = pd.read_csv('acq_params.tsv', sep='\t')
c = pd.read_csv('cohort.tsv', sep='\t')
m = c.merge(p, left_on='participant_id', right_on='sub')
m['protocol'] = m['serial'].astype(str) + '_tr' + m['tr'].astype(str) + '_sl' + m['nslices'].astype(str)
print(pd.crosstab(m['diagnosis'], m['protocol']), '\n')
print(pd.crosstab(m['diagnosis'], m['serial']), '\n')
from scipy.stats import chi2_contingency
print("protocol vs diagnosis chi2 p = %.4f" % chi2_contingency(pd.crosstab(m['diagnosis'], m['protocol']))[1])
m.to_csv('cohort_with_acq.tsv', sep='\t', index=False)

import numpy as np
b = np.loadtxt('raw/sub-50029/dwi/sub-50029_dwi.bval')
print(b.size, np.unique(np.round(b,-2)), 'n b0:', (b<50).sum())

protocol   35343_tr7.0_sl50  35343_tr8.4_sl60  35426_tr9.0_sl60
diagnosis                                                      
CONTROL                  12                29                 8
SCHZ                      1                24                24 

serial     35343  35426
diagnosis              
CONTROL       41      8
SCHZ          25     24 

protocol vs diagnosis chi2 p = 0.0001
32 [   0. 1000.] n b0: 1


In [7]:
# ---------------------------------------------------------------------------
# CELL 5 - Is the imbalance a property of the matched cohort, or of the whole
# eligible pool?
# ---------------------------------------------------------------------------

import pandas as pd
df = pd.read_csv('participants.tsv', sep='\t')
elig = df[df['diagnosis'].isin(['CONTROL','SCHZ']) & df['dwi'].notna()]
print(pd.crosstab(elig['diagnosis'], elig['ScannerSerialNumber']))

ScannerSerialNumber  35343.0  35426.0
diagnosis                            
CONTROL                  100       23
SCHZ                      25       24


In [8]:
# ---------------------------------------------------------------------------
# CELL 6 - Build the protocol table for EVERY eligible participant, including
# those not yet downloaded.
#
# The problem: protocol-aware matching needs protocol labels for the whole
# control pool, but downloading 130 controls to read six numbers from each
# sidecar would cost ~13 GB. Solution: fetch only the tiny .json sidecars into
# json_probe/, and merge them with the sidecars already present in raw/ for
# participants downloaded earlier.
# ---------------------------------------------------------------------------

import json, glob, pandas as pd, os
df = pd.read_csv('participants.tsv', sep='\t')

def proto(path):
    d = json.load(open(path))
    return f"{d['DeviceSerialNumber']}_tr{d['RepetitionTime']}_sl{len(d.get('SliceTiming',[]))}"

rows = []
for f in glob.glob('json_probe/*_dwi.json'):
    rows.append({'participant_id': os.path.basename(f).replace('_dwi.json',''),
                 'protocol': proto(f)})
for f in glob.glob('raw/sub-*/dwi/*_dwi.json'):
    rows.append({'participant_id': os.path.basename(f).replace('_dwi.json',''),
                 'protocol': proto(f)})

pr = pd.DataFrame(rows).drop_duplicates('participant_id')
m = df.merge(pr, on='participant_id')
m = m[m['diagnosis'].isin(['CONTROL','SCHZ']) & m['dwi'].notna()]
m = m[m['participant_id'] != 'sub-50029']

print(pd.crosstab([m['diagnosis'], m['gender']], m['protocol']))
m.to_csv('eligible_with_protocol.tsv', sep='\t', index=False)

protocol          35343_tr7_sl50  35343_tr8.4_sl60  35426_tr9_sl60
diagnosis gender                                                  
CONTROL   F                    7                39              12
          M                   13                41              11
SCHZ      F                    0                 7               4
          M                    1                16              20


In [9]:
# ---------------------------------------------------------------------------
# CELL 7 - FINAL cohort: match exactly on protocol AND sex, approximately on
# age.
#
# Identical algorithm to cell 2, with protocol added as a second exact-match
# constraint. Tightening the constraint costs participants: patients on a
# protocol with no available same-sex control cannot be matched at all.
# Writes cohort_final.tsv (the analysis cohort) and matched_pairs_final.tsv
# (the pairings, retained even though all analyses are unpaired).
# ---------------------------------------------------------------------------

import pandas as pd, numpy as np
from scipy import stats

m = pd.read_csv('eligible_with_protocol.tsv', sep='\t')
scz  = m[m['diagnosis']=='SCHZ'].copy()
ctrl = m[m['diagnosis']=='CONTROL'].copy()
scz = scz.reindex(scz['age'].sub(scz['age'].mean()).abs().sort_values(ascending=False).index)

used, pairs, dropped = set(), [], []
for _, p in scz.iterrows():
    cand = ctrl[(ctrl['gender']==p['gender']) & (ctrl['protocol']==p['protocol'])
                & (~ctrl['participant_id'].isin(used))]
    if cand.empty:
        dropped.append(p['participant_id']); continue
    c = cand.iloc[(cand['age']-p['age']).abs().argmin()]
    used.add(c['participant_id'])
    pairs.append({'scz_id':p['participant_id'],'ctrl_id':c['participant_id'],
                  'protocol':p['protocol'],'gender':p['gender'],
                  'scz_age':p['age'],'ctrl_age':c['age'],
                  'diff':abs(p['age']-c['age'])})

pairs = pd.DataFrame(pairs)
print(f"pairs: {len(pairs)}   dropped: {len(dropped)}")
print("age diff  mean %.2f  max %d\n" % (pairs['diff'].mean(), pairs['diff'].max()))

sel = m[m['participant_id'].isin(pairs['scz_id']) | m['participant_id'].isin(pairs['ctrl_id'])]
print(sel.groupby('diagnosis')['age'].agg(['count','mean','std']), '\n')
print(pd.crosstab(sel['diagnosis'], sel['protocol']), '\n')
print(pd.crosstab(sel['diagnosis'], sel['gender']), '\n')

a = sel[sel['diagnosis']=='SCHZ']['age']; b = sel[sel['diagnosis']=='CONTROL']['age']
print("age  p = %.3f" % stats.ttest_ind(a,b).pvalue)
print("prot p = %.3f" % stats.chi2_contingency(pd.crosstab(sel['diagnosis'],sel['protocol']))[1])
print("ghost:\n", pd.crosstab(sel['diagnosis'], sel['ghost_NoGhost']))

sel.to_csv('cohort_final.tsv', sep='\t', index=False)
pairs.to_csv('matched_pairs_final.tsv', sep='\t', index=False)
print("\ndropped:", dropped)

pairs: 39   dropped: 9
age diff  mean 1.67  max 11

           count       mean       std
diagnosis                            
CONTROL       39  35.025641  8.609931
SCHZ          39  35.923077  9.476799 

protocol   35343_tr7_sl50  35343_tr8.4_sl60  35426_tr9_sl60
diagnosis                                                  
CONTROL                 1                23              15
SCHZ                    1                23              15 

gender      F   M
diagnosis        
CONTROL    11  28
SCHZ       11  28 

age  p = 0.663
prot p = 1.000
ghost:
 ghost_NoGhost  No_ghost  ghost
diagnosis                     
CONTROL              32      7
SCHZ                 22     17

dropped: ['sub-50077', 'sub-50076', 'sub-50060', 'sub-50083', 'sub-50081', 'sub-50059', 'sub-50066', 'sub-50075', 'sub-50051']
